# DefinitionBot — Spec-definition workflow

Demonstrates the four-step flow that DefinitionBot uses internally:

```
record_definition_intent → list_tables / describe_table → draft_spec → validate_spec
```

**Part 1** (no API key): Direct YAML spec parsing — shows what DefinitionBot validates
before minting a `spec_draft_token`.  Each spec type is demonstrated with a valid YAML
string and its parsed attributes.

**Part 2** (requires API key): Two `DefinitionBot.ask()` calls using the ad campaign
dataset — one ratio metric (`avg_cpc`) and one wildcard slice (`country`).  Each run's
trace shows the four mandatory steps and the token-gating in action.

**Part 3** (Anthropic only): Prompt-cache efficiency.  Layer A (workflow rules) and
Layer B (existing catalog) are cached across `ask()` calls.

### Prerequisites

1. Set your Anthropic API key: `export ANTHROPIC_API_KEY=sk-ant-...`
2. Install the agent extra: `pip install aitaem[agent-anthropic]`
3. Set `aitaem_folder_path` below to the path of your local `aitaem` repo.

In [1]:
from __future__ import annotations

import os
import textwrap
from typing import Any

from aitaem.connectors import ConnectionManager
from aitaem.helpers import load_csvs_to_duckdb
from aitaem.specs import MetricSpec, SliceSpec, SpecCache
from aitaem.agent import (
    DefinitionBot,
    DefinitionResponse,
    RunTrace,
    Status,
)

Set the path to the local folder which contains your AITAEM examples. The following sub-folders are needed inside `aitaem_folder_path`:
- `examples/data/`
- `examples/metrics/`
- `examples/slices/`
- `examples/segments/`

In [ ]:
aitaem_folder_path = "/path/to/aitaem"  # e.g. "/Users/you/Workspace/aitaem"

## Setup — load specs and connect to DuckDB

In [3]:
# Full catalog — includes all metrics, slices, and segments.
# DefinitionBot checks this catalog in validate_spec to detect name conflicts.
spec_cache = SpecCache.from_yaml(
    metric_paths=aitaem_folder_path + "/examples/metrics/",
    slice_paths=aitaem_folder_path + "/examples/slices/",
    segment_paths=aitaem_folder_path + "/examples/segments/",
)
print(
    f"Catalog: {len(spec_cache.metrics)} metrics, "
    f"{len(spec_cache.slices)} slices, "
    f"{len(spec_cache.segments)} segment(s)"
)
print(f"  Metrics  : {', '.join(spec_cache.metrics)}")
print(f"  Slices   : {', '.join(spec_cache.slices)}")
print(f"  Segments : {', '.join(spec_cache.segments)}")

Catalog: 6 metrics, 3 slices, 1 segment(s)
  Metrics  : avg_revenue, campaign_count, ctr, max_revenue, roas, total_revenue
  Slices   : campaign_type, geo, industry
  Segments : platform


In [4]:
# Create DuckDB from the bundled CSV if it doesn't already exist.
# The connection is opened in Part 2 — Part 1 (direct parsing) needs no DB.
db_path = aitaem_folder_path + "/examples/data/ad_campaigns.duckdb"
if not os.path.exists(db_path):
    print("DuckDB file not found — creating from CSV …")
    load_csvs_to_duckdb(aitaem_folder_path + "/examples/data/ad_campaigns.csv", db_path)
    print(f"  Created {db_path}")
print("DuckDB path ready. Connection will open in Part 2.")

DuckDB path ready. Connection will open in Part 2.


### Helper functions

In [5]:
def _fmt_arg(v: Any) -> str:
    if isinstance(v, str) and len(v) > 14:
        return f"'{v[:10]}…'"
    if isinstance(v, list):
        return f"[{', '.join(repr(x) for x in v)}]"
    return repr(v)


def _print_spec_parse(label: str, yaml_string: str) -> None:
    """Parse a YAML string and print the resulting spec's attributes."""
    print(f"\n  {'─' * 62}")
    print(f"  {label}")
    print(f"  {'─' * 62}")
    top_key = yaml_string.strip().splitlines()[0].split(":")[0].strip()
    try:
        if top_key == "metric":
            spec = MetricSpec.from_yaml(yaml_string)
            print(f"  ✓  Parsed MetricSpec")
            print(f"     name        : {spec.name}")
            print(f"     source      : {spec.source}")
            print(f"     numerator   : {spec.numerator}")
            denom = getattr(spec, "denominator", None)
            if denom:
                print(f"     denominator : {denom}")
            print(f"     timestamp   : {spec.timestamp_col}")
            result = spec.validate()
            if result.referenced_columns:
                for field, cols in result.referenced_columns.items():
                    print(f"     refs [{field}]  : {cols}")
        elif top_key == "slice":
            spec = SliceSpec.from_yaml(yaml_string)
            print(f"  ✓  Parsed SliceSpec")
            print(f"     name        : {spec.name}")
            if spec.is_composite:
                print(f"     subtype     : composite")
                print(f"     cross_product: {spec.cross_product}")
            elif spec.is_wildcard:
                print(f"     subtype     : wildcard")
                print(f"     column      : {spec.column}")
            else:
                print(f"     subtype     : leaf ({len(spec.values)} values)")
                for sv in spec.values[:3]:
                    print(f"       - {sv.name}: {sv.where}")
                if len(spec.values) > 3:
                    print(f"       … {len(spec.values) - 3} more")
        else:
            print(f"  ✗  Unknown top-level key: {top_key!r}")
    except Exception as exc:
        print(f"  ✗  Parse failed: {type(exc).__name__}: {exc}")


def _print_response(label: str, response: DefinitionResponse) -> None:
    print(f"\n{'─' * 70}")
    print(f"{label}")
    print(f"{'─' * 70}")
    print(f"Status   : {response.status.value}")
    print(
        f"Narrative: {textwrap.fill(response.narrative, width=70, subsequent_indent='           ')}"
    )
    if response.status == Status.error and response.reason:
        print(f"Error    : {response.reason}")
        return
    if response.status == Status.refused:
        if response.reason:
            print(f"Reason   : {response.reason}")
        return
    p = response.payload
    if p is None or p.spec_type is None:
        return
    print(f"Spec     : {p.spec_type} / {p.spec_name}")
    print(f"Token    : {(p.spec_draft_token or '')[:16]}…")
    if p.referenced_columns:
        for src, cols in p.referenced_columns.items():
            print(f"Refs     : [{src}] {cols}")
    if p.validation_warnings:
        for w in p.validation_warnings:
            print(f"Warning  : {w}")


def _print_yaml(response: DefinitionResponse) -> None:
    p = response.payload
    if p is None or not p.yaml_string:
        return
    print(f"\nGenerated YAML:")
    for line in p.yaml_string.splitlines():
        print(f"  {line}")


def _print_trace(trace: RunTrace) -> None:
    print(
        f"\nTrace    : run={trace.run_id[:8]}  conv={trace.conversation_id[:8]}"
        f"  {trace.duration_ms:.0f}ms"
    )
    for tc in trace.tool_calls:
        non_null = {k: v for k, v in tc.args.items() if v is not None}
        args_str = ", ".join(f"{k}={_fmt_arg(v)}" for k, v in non_null.items())
        icon = "✓" if tc.success else "✗"
        print(f"  {icon} {tc.name}({args_str})")
    u = trace.usage
    cache_info = f"  cache_read={u.cache_read_tokens}" if u.cache_read_tokens else ""
    print(f"Tokens   : {u.input_tokens} in / {u.output_tokens} out{cache_info}")

---

## Part 1 — Direct spec parsing: what DefinitionBot validates (no API key)

`MetricSpec.from_yaml()` and `SliceSpec.from_yaml()` are the structural validators
that DefinitionBot's `validate_spec` tool calls internally at check #2.  This part
shows what valid YAML looks like for each spec type and what a structural parse
failure looks like — no LLM involved.

The two new specs we will define in Part 2:
- `avg_cpc` — a ratio metric: `SUM(ad_spend) / SUM(clicks)`
- `country` — a wildcard slice: auto-discovers country values from the column

### Scenario 1 — MetricSpec: `avg_cpc` (ratio metric)

A ratio metric has both `numerator` and `denominator`.  The parser extracts the SQL
expressions and calls `spec.validate()` to find the referenced column names.  These
columns are later checked against the live table schema in `validate_spec` check #5.

In [6]:
_print_spec_parse(
    "MetricSpec: avg_cpc — SUM(ad_spend) / SUM(clicks)",
    """\
metric:
  name: avg_cpc
  description: Average cost per click — total ad spend divided by total clicks
  source: duckdb://ad_campaigns.duckdb/ad_campaigns
  numerator: "SUM(ad_spend)"
  denominator: "SUM(clicks)"
  timestamp_col: date
  format: "currency:USD"
""",
)


  ──────────────────────────────────────────────────────────────
  MetricSpec: avg_cpc — SUM(ad_spend) / SUM(clicks)
  ──────────────────────────────────────────────────────────────
  ✓  Parsed MetricSpec
     name        : avg_cpc
     source      : duckdb://ad_campaigns.duckdb/ad_campaigns
     numerator   : SUM(ad_spend)
     denominator : SUM(clicks)
     timestamp   : date
     refs [numerator]  : ['ad_spend']
     refs [denominator]  : ['clicks']
     refs [timestamp_col]  : ['date']


### Scenario 2 — SliceSpec (wildcard): `country`

A wildcard slice uses a column name in the `where` field.  At query time the engine
runs `SELECT DISTINCT country FROM …` and groups dynamically.  No hard-coded values
are needed — ideal for high-cardinality dimensions that change over time.

In [7]:
_print_spec_parse(
    "SliceSpec (wildcard): country — auto-discovers values from column",
    """\
slice:
  name: country
  description: Breakdown by country — values auto-discovered from the country column
  where: country
""",
)


  ──────────────────────────────────────────────────────────────
  SliceSpec (wildcard): country — auto-discovers values from column
  ──────────────────────────────────────────────────────────────
  ✓  Parsed SliceSpec
     name        : country
     subtype     : wildcard
     column      : country


### Scenario 3 — Structural parse failure

`numerator` is required for MetricSpec.  Omitting it triggers a `SpecValidationError`
at check #2 of `validate_spec`.  DefinitionBot returns this as a `ValidationIssue` list
so the LLM can fix the draft and call `draft_spec` again with the corrected YAML.

In [8]:
_print_spec_parse(
    "MetricSpec parse failure: missing required 'numerator' field",
    """\
metric:
  name: bad_metric
  source: duckdb://ad_campaigns.duckdb/ad_campaigns
""",
)


  ──────────────────────────────────────────────────────────────
  MetricSpec parse failure: missing required 'numerator' field
  ──────────────────────────────────────────────────────────────
  ✗  Parse failed: SpecValidationError: Invalid metric spec 'bad_metric':
  - Field 'numerator': 'numerator' must be a non-empty SQL expression
  - Field 'timestamp_col': 'timestamp_col' is required and must be a non-empty string (suggestion: Add the date/timestamp column name used for time filtering, e.g. 'timestamp_col: created_at')


---

## Part 2 — DefinitionBot: four-step flow in a real session

Each `ask()` call's trace must show:
1. `record_definition_intent` — LLM records what spec to create and its type
2. `list_tables` + `describe_table` — schema exploration (no column invention)
3. `draft_spec` — stores LLM-written YAML under a `draft_id` (no validation yet)
4. `validate_spec` — runs five checks; mints `spec_draft_token` on full pass

The LLM cannot return a non-null `spec_draft_token` without a successful `validate_spec`
result — the token is the anti-hallucination gate.

In [9]:
# Close any previous DuckDB connection before re-creating the manager.
# Re-running this cell without closing would hold two locks on the same file.
try:
    conn_mgr.close_all()
except NameError:
    pass  # first run — conn_mgr not yet defined

conn_mgr = ConnectionManager()
conn_mgr.add_connection("duckdb", path=db_path)

bot = DefinitionBot(
    model="anthropic:claude-haiku-4-5-20251001",
    spec_cache=spec_cache,
    connection_manager=conn_mgr,
)

### Ask 1 — ratio metric: `avg_cpc`

The bot must call `list_tables`, then `describe_table` to confirm `ad_spend` and
`clicks` exist before drafting the YAML.  It then calls `validate_spec` which checks
structural validity, name conflict (against existing catalog), and column existence.
On full pass a `spec_draft_token` is minted and returned in `response.payload`.

This call also **warms Layers A and B** in Anthropic's prompt cache.

In [10]:
response1 = await bot.ask(
    "Define a metric called avg_cpc for average cost per click — "
    "total ad spend divided by total clicks."
)
_print_response("Ask 1: avg_cpc — average cost per click", response1)
_print_yaml(response1)
_print_trace(response1.trace)


──────────────────────────────────────────────────────────────────────
Ask 1: avg_cpc — average cost per click
──────────────────────────────────────────────────────────────────────
Status   : ok
Narrative: Successfully defined metric **avg_cpc** (Average Cost Per Click). This
           ratio metric divides total ad spend (numerator) by total
           clicks (denominator) to compute the average cost incurred
           per click. The metric is sourced from the ad_campaigns
           table, uses the date column for time-based aggregation, and
           is formatted as USD currency. All referenced columns
           (ad_spend, clicks, date) exist in the schema.
Spec     : metric / avg_cpc
Token    : 85737136-7a21-4f…
Refs     : [numerator] ['ad_spend']
Refs     : [denominator] ['clicks']
Refs     : [timestamp_col] ['date']

Generated YAML:
  metric:
    name: avg_cpc
    description: Average cost per click — total ad spend divided by total clicks
    source: duckdb://ad_campaigns.d

### Ask 2 — wildcard slice: `country`

A wildcard slice only needs a column name in `where`.  The bot still calls
`describe_table` to confirm the column exists, then drafts and validates the YAML.
Layer A+B should now be **read from Anthropic's cache**.

In [11]:
response2 = await bot.ask(
    "Define a wildcard slice called country that auto-discovers "
    "all country values from the country column in the campaigns table."
)
_print_response("Ask 2: country — wildcard slice", response2)
_print_yaml(response2)
_print_trace(response2.trace)


──────────────────────────────────────────────────────────────────────
Ask 2: country — wildcard slice
──────────────────────────────────────────────────────────────────────
Status   : ok
Narrative: Successfully defined a wildcard slice called "country" that auto-
           discovers all unique country values from the country column
           in the ad_campaigns table. The slice will dynamically
           generate slice members based on distinct values in the
           country column, allowing breakdowns by each unique country
           present in the data.
Spec     : slice / country
Token    : 87034789-224c-4c…
Refs     : [where] ['country']

Generated YAML:
  slice:
    name: country
    description: Auto-discover all country values from the campaigns table
    where: country

Trace    : run=019f4451  conv=019f4451  6591ms
  ✓ record_definition_intent(spec_type='slice', description='Auto-disco…')
  ✓ list_tables()
  ✓ describe_table(table_name='ad_campaigns', backend_type='duck

---

## Part 3 — Prompt-cache efficiency (Anthropic only)

DefinitionBot v1 uses the same three-layer system prompt as QueryBot:

| Layer | Content | Cached? |
|-------|---------|--------|
| A | Workflow rules and YAML format reference (identical for all tenants) | Yes |
| B | Per-tenant existing catalog (metric / slice / segment names and details) | Yes |
| C | Today's date (changes each calendar day) | No |

After the first `ask()` warms Layers A+B, subsequent calls should show
`cache_read_tokens > 0` in the usage block.

In [12]:
def _cache_summary(label: str, response: Any) -> None:
    u = response.trace.usage
    ct = u.cache_read_tokens or 0
    status = "✓  served from cache" if ct > 0 else "✗  no cache hit"
    print(f"{label}")
    print(f"  input_tokens       : {u.input_tokens:>6}")
    print(f"  cache_read_tokens  : {ct:>6}  {status}")
    print(f"  output_tokens      : {u.output_tokens:>6}")
    print()


if "anthropic:" in bot._model:
    print("Cache efficiency summary (Anthropic only)")
    print("─" * 50)
    _cache_summary("Ask 1 (warms cache — no read expected)", response1)
    _cache_summary("Ask 2 (Layers A+B should be cached)", response2)
else:
    print(f"Cache check skipped — provider is not Anthropic (model={bot._model!r}).")

Cache efficiency summary (Anthropic only)
──────────────────────────────────────────────────
Ask 1 (warms cache — no read expected)
  input_tokens       :  24934
  cache_read_tokens  :   8613  ✓  served from cache
  output_tokens      :    543

Ask 2 (Layers A+B should be cached)
  input_tokens       :  20897
  cache_read_tokens  :   8554  ✓  served from cache
  output_tokens      :    451

